# Hall-GMD ML Virtual Reference

Full-spectrum virtual-reference workflow: train on no-sample run 58780, predict a shot-level virtual I0 spectrum for sample run 58794, then bin/sum/compute absorbance.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
ML_ROOT = None
for p in [cwd] + list(cwd.parents):
    if (p / 'pipeline_config.py').exists() and (p / 'src' / 'static_xas_ml').exists():
        ML_ROOT = p
        break
    if (p / 'ML' / 'pipeline_config.py').exists():
        ML_ROOT = p / 'ML'
        break
if ML_ROOT is None:
    raise RuntimeError('Could not locate ML root.')
sys.path.insert(0, str(ML_ROOT))
sys.path.insert(0, str(ML_ROOT / 'src'))
print(ML_ROOT)

In [ ]:
RUN_PIPELINE = False
RUN_NAME = 'hall_virtual_reference_58780_to_58794'

if RUN_PIPELINE:
    cmd = [sys.executable, str(ML_ROOT / 'scripts' / 'run_pipeline.py'), '--name', RUN_NAME]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

RESULTS_DIR = ML_ROOT / 'results' / RUN_NAME
print(RESULTS_DIR)
if not RESULTS_DIR.exists():
    print('No results yet. Set RUN_PIPELINE = True, or run ML/scripts/run_pipeline.py --name ' + RUN_NAME)

In [ ]:
import pandas as pd
from IPython.display import Image, display

def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        print(f'missing: {path}')
        return pd.DataFrame()
    return pd.read_csv(path)

filter_counts = read_csv_if_exists(RESULTS_DIR / 'metrics' / 'filter_survival_counts.csv')
heldout = read_csv_if_exists(RESULTS_DIR / 'metrics' / 'heldout_reference_metrics.csv')
coverage = read_csv_if_exists(RESULTS_DIR / 'metrics' / 'feature_coverage_reference_vs_sample.csv')
baseline = read_csv_if_exists(RESULTS_DIR / 'metrics' / 'baseline_slope_checks.csv')
nominal = read_csv_if_exists(RESULTS_DIR / 'metrics' / 'binned_ml_absorbance_nominal.csv')
converted = read_csv_if_exists(RESULTS_DIR / 'metrics' / 'binned_ml_absorbance_converted.csv')

display(filter_counts)
display(heldout)
display(coverage)
display(baseline)
display(nominal.head())
display(converted.head())

In [ ]:
plot_names = [
    'pca_explained_variance.png',
    'heldout_reference_predicted_vs_true.png',
    'heldout_reference_error_histograms.png',
    'feature_coverage_reference_vs_sample.png',
    'nominal_energy_binned_ml_absorbance.png',
    'converted_energy_binned_ml_absorbance.png',
]
for name in plot_names:
    path = RESULTS_DIR / 'plots' / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))